In [54]:
import pandas as pd
import datetime as dt

In [27]:
url = 'OnlineRetail1.csv'
df = pd.read_csv(url)

In [28]:
df.fillna(value= '3.6')
df = df[pd.to_numeric(df['InvoiceNo'], errors='coerce').notnull()]
df['InvoiceNo'] = df['InvoiceNo'].astype('float32')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 199656 entries, 0 to 203421
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    199656 non-null  float32       
 1   StockCode    199656 non-null  object        
 2   Description  198857 non-null  object        
 3   Quantity     199656 non-null  int64         
 4   InvoiceDate  199656 non-null  datetime64[ns]
 5   UnitPrice    199656 non-null  float64       
 6   CustomerID   146488 non-null  float64       
 7   Country      199656 non-null  object        
dtypes: datetime64[ns](1), float32(1), float64(2), int64(1), object(3)
memory usage: 12.9+ MB


In [46]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total,Monetary,Money Value
0,539993.0,22386,JUMBO BAG PINK POLKADOT,10,2011-01-04 10:00:00,1.95,13313.0,United Kingdom,19.50,NaN,False
1,539993.0,21499,BLUE POLKADOT WRAP,25,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,10.50,NaN,False
2,539993.0,21498,RED RETROSPOT WRAP,25,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,10.50,NaN,False
3,539993.0,22379,RECYCLING BAG RETROSPOT,5,2011-01-04 10:00:00,2.10,13313.0,United Kingdom,10.50,NaN,False
4,539993.0,20718,RED RETROSPOT SHOPPER BAG,10,2011-01-04 10:00:00,1.25,13313.0,United Kingdom,12.50,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
203417,558637.0,22032,BOTANICAL LILY GREETING CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,NaN,False
203418,558637.0,22028,PENNY FARTHING BIRTHDAY CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,NaN,False
203419,558637.0,22033,BOTANICAL ROSE GREETING CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,NaN,False
203420,558637.0,22029,SPACEBOY BIRTHDAY CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,NaN,False


In [53]:
df['total'] = df['Quantity']*df['UnitPrice']
df['Monetary'] = df.groupby('CustomerID')['total'].transform('sum')
df['Money Value'] = df['Monetary'] >1000
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total,Monetary,Money Value
0,539993.0,22386,JUMBO BAG PINK POLKADOT,10,2011-01-04 10:00:00,1.95,13313.0,United Kingdom,19.50,609.74,False
1,539993.0,21499,BLUE POLKADOT WRAP,25,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,10.50,609.74,False
2,539993.0,21498,RED RETROSPOT WRAP,25,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,10.50,609.74,False
3,539993.0,22379,RECYCLING BAG RETROSPOT,5,2011-01-04 10:00:00,2.10,13313.0,United Kingdom,10.50,609.74,False
4,539993.0,20718,RED RETROSPOT SHOPPER BAG,10,2011-01-04 10:00:00,1.25,13313.0,United Kingdom,12.50,609.74,False
...,...,...,...,...,...,...,...,...,...,...,...
203417,558637.0,22032,BOTANICAL LILY GREETING CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,203.86,False
203418,558637.0,22028,PENNY FARTHING BIRTHDAY CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,203.86,False
203419,558637.0,22033,BOTANICAL ROSE GREETING CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,203.86,False
203420,558637.0,22029,SPACEBOY BIRTHDAY CARD,12,2011-06-30 20:08:00,0.42,17891.0,United Kingdom,5.04,203.86,False


In [56]:
reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
recency = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
recency['Recency'] = (reference_date - recency['InvoiceDate']).dt.days
frequency = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
frequency.columns = ['CustomerID', 'Frequency']
monetary = df.groupby('CustomerID')['total'].sum().reset_index()
monetary.columns = ['CustomerID', 'Monetary']
rfm = recency.merge(frequency, on='CustomerID')
rfm = rfm.merge(monetary, on='CustomerID')
rfm.head()


,CustomerID,InvoiceDate,Recency,Frequency,Monetary
0,12346.0,2011-01-18 10:01:00,164,1,77183.60
1,12347.0,2011-06-09 13:01:00,22,3,1494.16
2,12348.0,2011-04-05 10:47:00,87,2,594.44
3,12350.0,2011-02-02 16:01:00,149,1,334.40
4,12352.0,2011-03-22 16:08:00,101,5,1561.81


In [ ]:
# Hàm tính điểm
def r_score(x,p,d): 
    if x <= d[p][0.25]: return 4
    elif x <= d[p][0.50]: return 3
    elif x <= d[p][0.75]: return 2
    else: return 1

def fm_score(x,p,d): 
    if x <= d[p][0.25]: return 1
    elif x <= d[p][0.50]: return 2
    elif x <= d[p][0.75]: return 3
    else: return 4

# Tính quantile
quantiles = rfm.quantile(q=[0.25,0.5,0.75]).to_dict()

# Gán điểm
rfm['R_Score'] = rfm['Recency'].apply(r_score, args=('Recency',quantiles))
rfm['F_Score'] = rfm['Frequency'].apply(fm_score, args=('Frequency',quantiles))
rfm['M_Score'] = rfm['Monetary'].apply(fm_score, args=('Monetary',quantiles))

# Tạo RFM Score
rfm['RFM_Score'] = rfm['R_Score'].map(str) + rfm['F_Score'].map(str) + rfm['M_Score'].map(str)
